In [1]:
import re
import math
from collections import defaultdict, Counter

# Dataset
documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom how are you", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off limited time", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes", "SPAM"),
    ("Can you send the report", "HAM"),
]

In [2]:
#Preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

In [3]:
#Generate Bag of Words
def generate_bow(docs):
    vocab = set()
    word_counts = {
        "SPAM": defaultdict(int),
        "HAM": defaultdict(int)
    }
    class_counts = {"SPAM": 0, "HAM": 0}
    
    for text, label in docs:
        class_counts[label] += 1
        words = preprocess(text)
        
        for word in words:
            vocab.add(word)
            word_counts[label][word] += 1
    
    return list(vocab), word_counts, class_counts

In [4]:
#Compute Priors
def compute_priors(class_counts):
    total_docs = sum(class_counts.values())
    
    priors = {}
    for label in class_counts:
        priors[label] = class_counts[label] / total_docs
    
    return priors

In [5]:
#Compute Likelihoods
def compute_likelihoods(vocab, word_counts):
    likelihoods = {
        "SPAM": {},
        "HAM": {}
    }
    
    V = len(vocab)
    
    for label in ["SPAM", "HAM"]:
        total_words = sum(word_counts[label].values())
        
        for word in vocab:
            likelihoods[label][word] = (
                word_counts[label][word] + 1
            ) / (total_words + V)
    
    return likelihoods

In [6]:
#Classify Sentence
def classify(sentence, vocab, priors, likelihoods, word_counts):
    words = preprocess(sentence)
    scores = {}
    V = len(vocab)
    
    for label in ["SPAM", "HAM"]:
        log_prob = math.log(priors[label])
        total_words = sum(word_counts[label].values())
        
        for word in words:
            if word in vocab:
                log_prob += math.log(likelihoods[label][word])
            else:
                # handle unseen words
                log_prob += math.log(1 / (total_words + V))
        
        scores[label] = log_prob
    
    return max(scores, key=scores.get)

In [7]:
# Generate Bag of Words
vocab, word_counts, class_counts = generate_bow(documents)

# Compute Priors
priors = compute_priors(class_counts)

# Compute Likelihoods
likelihoods = compute_likelihoods(vocab, word_counts)

# Test Sentences
test1 = "Limited offer click here"
test2 = "Meeting at 2 PM with the manager"

print("Test 1 Prediction:", classify(test1, vocab, priors, likelihoods, word_counts))
print("Test 2 Prediction:", classify(test2, vocab, priors, likelihoods, word_counts))

Test 1 Prediction: SPAM
Test 2 Prediction: HAM


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Dataset
texts = [doc for doc, label in documents]
labels = [label for doc, label in documents]

# Vectorize the text data
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# Train the Multinomial Naïve Bayes classifier
clf = MultinomialNB()
clf.fit(X, labels)

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager"
]

# Transform test sentences into feature vectors
X_test = vectorizer.transform(test_sentences)

# Predict classes
predictions = clf.predict(X_test)

# Display results
for sentence, pred in zip(test_sentences, predictions):
    print(f"Sentence: '{sentence}' → Prediction: {pred}")


Sentence: 'Limited offer, click here!' → Prediction: SPAM
Sentence: 'Meeting at 2 PM with the manager' → Prediction: HAM


In [9]:
import pandas as pd

# Manual predictions (from your earlier classify function)
manual_preds = [
    classify("Limited offer click here", vocab, priors, likelihoods, word_counts),
    classify("Meeting at 2 PM with the manager", vocab, priors, likelihoods, word_counts)
]

# Scikit-Learn predictions (already computed)
sklearn_preds = predictions

# Build comparison table
df = pd.DataFrame({
    "Sentence": test_sentences,
    "Manual Prediction": manual_preds,
    "Scikit-Learn Prediction": sklearn_preds
})

print(df)


                           Sentence Manual Prediction Scikit-Learn Prediction
0        Limited offer, click here!              SPAM                    SPAM
1  Meeting at 2 PM with the manager               HAM                     HAM
